<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/DQN_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import random
from collections import deque
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

In [ ]:
pip install gymnasium

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gymnasium as gym

In [ ]:
# ===== ENVIRONMENT =====
env = gym.make("CartPole-v1")
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

# ===== HYPERPARAMETERS =====
gamma = 0.95          # discount factor
epsilon = 1.0         # exploration rate
epsilon_min = 0.01
epsilon_decay = 0.995
learning_rate = 0.001
batch_size = 32

memory = deque(maxlen=2000)

In [ ]:
# ===== DQN AGENT =====
def build_model():
    model = Sequential()
    model.add(Dense(24, input_dim=state_size, activation='relu'))
    model.add(Dense(24, activation='relu'))
    model.add(Dense(action_size, activation='linear'))
    model.compile(loss='mse', optimizer=Adam(learning_rate=learning_rate))
    return model

model = build_model()

C:\Users\Darrick\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# ===== STORE EXPERIENCE =====
def remember(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))


# ===== ACTION SELECTION =====
def act(state):
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)  # explore
    q_values = model.predict(state, verbose=0)
    return np.argmax(q_values[0])  # exploit

In [ ]:
# ===== TRAINING =====
def replay():
    global epsilon
    minibatch = random.sample(memory, batch_size)

    for state, action, reward, next_state, done in minibatch:
        target = reward
        if not done:
            target = reward + gamma * np.max(model.predict(next_state, verbose=0)[0])

        target_f = model.predict(state, verbose=0)
        target_f[0][action] = target
        model.fit(state, target_f, epochs=1, verbose=0)

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

In [ ]:
# ===== MAIN TRAINING LOOP =====
episodes = 100

for e in range(episodes):
    state = env.reset()[0]
    state = np.reshape(state, [1, state_size])
    total_reward = 0

    for time in range(500):
        action = act(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state = np.reshape(next_state, [1, state_size])

        remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

        if done:
            print(f"Episode {e+1}/{episodes} - Score: {total_reward}")
            break

        if len(memory) > batch_size:
            replay()

env.close()

Episode 1/100 - Score: 15.0
Episode 2/100 - Score: 10.0
Episode 3/100 - Score: 21.0
Episode 4/100 - Score: 10.0
Episode 5/100 - Score: 18.0
Episode 6/100 - Score: 13.0
Episode 7/100 - Score: 11.0
Episode 8/100 - Score: 10.0
Episode 9/100 - Score: 11.0
Episode 10/100 - Score: 11.0
Episode 11/100 - Score: 10.0
Episode 12/100 - Score: 21.0
Episode 13/100 - Score: 14.0
Episode 14/100 - Score: 10.0
Episode 15/100 - Score: 77.0
Episode 16/100 - Score: 41.0
Episode 17/100 - Score: 72.0
Episode 18/100 - Score: 44.0
Episode 19/100 - Score: 54.0
Episode 20/100 - Score: 57.0
